In [1]:
#pip install chess
import chess
import math
import random
import multiprocessing
from multiprocessing import Pool
from multiprocessing.dummy import Pool as ThreadPool

**UCT formula ("Upper Confidence Bound applied to Trees")** is the core selection policy in Monte Carlo Tree Search (MCTS). It balances two conflicting goals:
* Exploration: Trying out less-visited, potentially promising moves -> `c_param * sqrt(2 * log(parent.visits) / child.visits)`
* Exploitation: Focusing on the moves that have performed well so far -> `child.wins / child.visits`

UCT = Exploitation (mean reward) + Exploration (uncertainty) encourages the search to both "exploit the best" and "explore the unknown"

In [ ]:
class GameState:
    """
    Class representing the state of the chess game.
    It encapsulates the chess board and provides methods to get possible moves,
    make moves, check if the game is over, and evaluate the board.
    """
    def __init__(self, board=None):
        self.board = board if board is not None else chess.Board()
    
    def get_possible_moves(self):
        return list(self.board.legal_moves)
    
    def make_move(self, move):
        if move not in self.board.legal_moves:
            raise ValueError(f"Illegal move: {move.uci()}")
        new_board = self.board.copy()
        new_board.push(move)
        return GameState(new_board)
    
    def is_terminal(self):
        return self.board.is_game_over()
    
    def get_reward(self):
        # Reward: +1 (bot wins), -1 (bot loses), 0 (draw), intermediate: material eval
        if self.board.is_checkmate():
            # if it's black's turn and game over, white won
            return 1 if self.board.turn == chess.BLACK else -1
        elif self.board.is_stalemate() or self.board.is_insufficient_material():
            return 0
        else:
            return self.evaluate_board()
    
    def evaluate_board(self):
        values = {
            chess.PAWN: 1,
            chess.KNIGHT: 3,
            chess.BISHOP: 3,
            chess.ROOK: 5,
            chess.QUEEN: 9,
            chess.KING: 0,
        }
        material_score = sum(
            (values[piece.piece_type] if piece.color == chess.WHITE else -values[piece.piece_type])
            for piece in self.board.piece_map().values()
        )
        return material_score
    
# MCTS Tree Node
class Node:
    def __init__(self, state, parent=None):
        self.state = state         # GameState object
        self.parent = parent
        self.children = []
        self.visits = 0
        self.wins = 0
        self.untried_moves = state.get_possible_moves()
        self.player_turn = state.board.turn
        self.is_terminal = state.is_terminal()
    
    def is_fully_expanded(self):
        """
        Check if all possible moves have been tried.
        """
        return len(self.children) == len(self.state.get_possible_moves())
    
    def best_child(self, c_param=1.4, h_weight=1.0):
        """
        Select the child node with the highest UCT value.
        c_param: exploration parameter
        h_weight: heuristic weight
        """
        my_board = self.state.board
        choices_weights = []
        for child in self.children:
            move = child.state.board.peek()  # the move that led to this child
            # Add board-aware heuristic bias, scaled by h_weight
            hval = move_heuristic(my_board, move)
            exploitation = child.wins / (child.visits + 1e-8)
            exploration = c_param * math.sqrt((2 * math.log(self.visits + 1) / (child.visits + 1e-8)))
            choices_weights.append(exploitation + exploration + h_weight * hval)
        return self.children[choices_weights.index(max(choices_weights))]


# MCTS Class
class MCTS:
    """
    Monte Carlo Tree Search (MCTS) algorithm for game AI.
    This class implements the MCTS algorithm with a focus on chess.
    """
    def __init__(self, mcts_iterations=100):
        self.mcts_iterations = mcts_iterations
    
    def search(self, initial_state):
        """
        Perform MCTS search on the initial state.
        initial_state: GameState object representing the initial state of the game.
        Returns the best move found after mcts_iterations iterations.
        """
        root = Node(initial_state)
        for iteration in range(self.mcts_iterations):
            c_param = self.dynamic_c_param(iteration)
            node = self.select(root, c_param)
            reward = self.simulate(node.state)
            self.backpropagate(node, reward)
        #return root.best_child(0).state  # c_param=0 => pure exploitation
        return root.best_child(0, h_weight=0.75).state # h_weight=0.75 for heuristic bias

    def dynamic_c_param(self, iteration):
        """
        Dynamically adjust the exploration parameter based on the iteration count.
        This is a simple linear decay function.
        """
        max_c_param, min_c_param = 1.4, 0.1
        return max(min_c_param, max_c_param * (1 - iteration / self.mcts_iterations))

    def select(self, node, c_param):
        """
        Select a node to expand using UCT (Upper Confidence Bound for Trees).
        c_param: exploration parameter
        """
        while not node.state.is_terminal():
            if not node.is_fully_expanded():
                return self.expand(node)
            else:
                #node = node.best_child(c_param) # pure UCT
                node = node.best_child(c_param, h_weight=0.5) # heuristic bias
        return node

    def expand(self, node):
        """
        Expand the node by adding a child node for an untried move.
        """
        tried_moves = [child.state.board.peek() for child in node.children if node.children]
        untried_moves = [move for move in node.state.get_possible_moves() if move not in tried_moves]
        move = random.choice(untried_moves)
        new_state = node.state.make_move(move)
        child_node = Node(new_state, node)
        node.children.append(child_node)
        return child_node

    def simulate(self, state):
        """
        Simulate a random game from the given state until a terminal state is reached.
        Returns the reward of the terminal state.
        """
        current_state = state
        while not current_state.is_terminal():
            possible_moves = current_state.get_possible_moves()
            move = random.choice(possible_moves)
            current_state = current_state.make_move(move)
        return current_state.get_reward()

    def backpropagate(self, node, reward):
        """
        Backpropagate the result of the simulation up the tree.
        node: Node object representing the node to backpropagate from.
        reward: Reward received from the simulation.
        """
        while node:
            node.visits += 1
            node.wins += reward
            node = node.parent

    def simulate_multiple_moves_parallel(self, game_state, mcts_iterations, depth=3):
        """
        Simulate multiple moves in parallel using threading.
        Args:
            game_state: The current game state.
            mcts_iterations: Number of MCTS iterations for each move.
            depth: Depth of the simulation.
        Returns:
            A list of dictionaries containing the results of each move simulation.
        """
        legal_moves = list(game_state.board.legal_moves)
        args = [(move, game_state, mcts_iterations, depth) for move in legal_moves]
        with ThreadPool() as pool:
            results = pool.map(simulate_single_move, args)
        return results
    

def simulate_single_move(args):
    """
    Simulate a single move and return the result.
    Args:
        args: Tuple containing the move, game state, MCTS iterations, and depth.
    Returns:
        A dictionary containing the move, sequence of moves, reward, terminal state, and board state.
    """

    move, game_state, mcts_iterations, depth = args
    sequence = [move.uci()]
    try:
        current_state = game_state.make_move(move)
    except ValueError as e:
        return {
            "move": move.uci(),
            "sequence": [],
            "reward": None,
            "is_terminal": False,
            "board": game_state.board.fen(),
            "error": str(e)
        }
    mcts = MCTS(mcts_iterations)
    for _ in range(depth - 1):
        if current_state.is_terminal():
            break
        best_state = mcts.search(current_state)
        best_move = best_state.board.peek()
        current_state = best_state
        sequence.append(best_move.uci())
    return {
        "move": move.uci(),
        "sequence": sequence,
        "reward": current_state.get_reward(),
        "is_terminal": current_state.is_terminal(),
        "board": current_state.board.fen(),
    }


def move_heuristic(board: chess.Board, move: chess.Move) -> float:
        """ 
        Heuristic = reward if move is a 'good' capture (captures more valuable piece than own),
        penalty if moving piece will be immediately taken by opponent.
        """
        values = {
            chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
            chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
        }

        # Check if move is a capture
        captured_piece = board.piece_at(move.to_square)
        moved_piece = board.piece_at(move.from_square)
        score = 0.0

        # Heuristic 1: Favor up-trades
        if captured_piece and moved_piece:
            if values[captured_piece.piece_type] > values[moved_piece.piece_type]:
                score += values[captured_piece.piece_type] - values[moved_piece.piece_type]  # positive bias
            else:
                score -= 0.5  # discourage low-value or even trades (else =0)
        
        # Do a shallow safety check: see if destination is attacked after the move
        temp_board = board.copy()
        temp_board.push(move)
        moved_color = moved_piece.color if moved_piece else None

        # If the destination square (now with our piece) is attacked by any opponent piece
        if moved_piece and temp_board.is_attacked_by(not moved_color, move.to_square):
            # Penalty is proportional to value of moved piece
            score -= values[moved_piece.piece_type]
        return score

In [ ]:
if __name__ == "__main__":
    # Initialize the game state
    game_state = GameState()
    mcts_iterations = 10
    mcts = MCTS()

    # Simulate multiple moves in parallel
    results = mcts.simulate_multiple_moves_parallel(game_state, mcts_iterations, depth=3)

    # Print the consequences of each move
    for result in results:
        print(f"Move: {result['move']}")
        print(f"  Sequence: {' -> '.join(result['sequence'])}")
        print(f"  Reward: {result['reward']}")
        print(f"  Is Terminal: {result['is_terminal']}")
        print()

Move: g1h3
  Sequence: g1h3 -> g8h6 -> f2f4
  Reward: 0
  Is Terminal: False

Move: g1f3
  Sequence: g1f3 -> f7f6 -> b1a3
  Reward: 0
  Is Terminal: False

Move: b1c3
  Sequence: b1c3 -> h7h5 -> f2f4
  Reward: 0
  Is Terminal: False

Move: b1a3
  Sequence: b1a3 -> d7d6 -> c2c3
  Reward: 0
  Is Terminal: False

Move: h2h3
  Sequence: h2h3 -> b8c6 -> d2d3
  Reward: 0
  Is Terminal: False

Move: g2g3
  Sequence: g2g3 -> d7d6 -> g1f3
  Reward: 0
  Is Terminal: False

Move: f2f3
  Sequence: f2f3 -> a7a6 -> b1a3
  Reward: 0
  Is Terminal: False

Move: e2e3
  Sequence: e2e3 -> b8a6 -> g2g3
  Reward: 0
  Is Terminal: False

Move: d2d3
  Sequence: d2d3 -> e7e5 -> f2f4
  Reward: 0
  Is Terminal: False

Move: c2c3
  Sequence: c2c3 -> b7b5 -> e2e3
  Reward: 0
  Is Terminal: False

Move: b2b3
  Sequence: b2b3 -> c7c6 -> b1a3
  Reward: 0
  Is Terminal: False

Move: a2a3
  Sequence: a2a3 -> b8a6 -> e2e3
  Reward: 0
  Is Terminal: False

Move: h2h4
  Sequence: h2h4 -> a7a6 -> b2b4
  Reward: 0
  Is Ter

In [10]:
if __name__ == "__main__":
    # Initialize the game state
    game_state = GameState()
    mcts_iterations = 10
    mcts = MCTS()

    # Simulate multiple moves in parallel
    results = mcts.simulate_multiple_moves_parallel(game_state, mcts_iterations, depth=5)

    # Print the consequences of each move
    for result in results:
        print(f"Move: {result['move']}")
        print(f"  Sequence: {' -> '.join(result['sequence'])}")
        print(f"  Reward: {result['reward']}")
        print(f"  Is Terminal: {result['is_terminal']}")
        print()

Move: g1h3
  Sequence: g1h3 -> g7g5 -> c2c4 -> b7b6 -> h3f4
  Reward: 0
  Is Terminal: False

Move: g1f3
  Sequence: g1f3 -> b8c6 -> e2e3 -> f7f6 -> d1e2
  Reward: 0
  Is Terminal: False

Move: b1c3
  Sequence: b1c3 -> c7c5 -> a2a3 -> b8a6 -> a1b1
  Reward: 0
  Is Terminal: False

Move: b1a3
  Sequence: b1a3 -> h7h6 -> d2d4 -> d7d5 -> d1d3
  Reward: 0
  Is Terminal: False

Move: h2h3
  Sequence: h2h3 -> g7g5 -> g2g3 -> a7a5 -> g1f3
  Reward: 0
  Is Terminal: False

Move: g2g3
  Sequence: g2g3 -> b7b5 -> g3g4 -> e7e5 -> f1g2
  Reward: 0
  Is Terminal: False

Move: f2f3
  Sequence: f2f3 -> h7h6 -> d2d4 -> g7g5 -> e2e3
  Reward: 0
  Is Terminal: False

Move: e2e3
  Sequence: e2e3 -> b7b5 -> a2a3 -> c8a6 -> d1e2
  Reward: 0
  Is Terminal: False

Move: d2d3
  Sequence: d2d3 -> e7e6 -> e2e3 -> d7d5 -> d1g4
  Reward: 0
  Is Terminal: False

Move: c2c3
  Sequence: c2c3 -> b8a6 -> c3c4 -> d7d5 -> b2b3
  Reward: 0
  Is Terminal: False

Move: b2b3
  Sequence: b2b3 -> g7g6 -> c2c3 -> c7c5 -> g1f3


In [11]:
if __name__ == "__main__":
    # Initialize the game state
    game_state = GameState()
    mcts_iterations = 10
    mcts = MCTS()

    # Simulate multiple moves in parallel
    results = mcts.simulate_multiple_moves_parallel(game_state, mcts_iterations, depth=10)

    # Print the consequences of each move
    for result in results:
        print(f"Move: {result['move']}")
        print(f"  Sequence: {' -> '.join(result['sequence'])}")
        print(f"  Reward: {result['reward']}")
        print(f"  Is Terminal: {result['is_terminal']}")
        print()

Move: g1h3
  Sequence: g1h3 -> a7a6 -> a2a4 -> d7d6 -> a4a5 -> b7b5 -> a1a3 -> h7h5 -> e2e4 -> b5b4
  Reward: 0
  Is Terminal: False

Move: g1f3
  Sequence: g1f3 -> e7e5 -> f3e5 -> d7d6 -> e5d3 -> g8f6 -> b2b4 -> e8d7 -> a2a3 -> f6g8
  Reward: 1
  Is Terminal: False

Move: b1c3
  Sequence: b1c3 -> g7g6 -> c3b1 -> b7b5 -> d2d4 -> h7h5 -> c1d2 -> c7c6 -> d2f4 -> b8a6
  Reward: 0
  Is Terminal: False

Move: b1a3
  Sequence: b1a3 -> e7e5 -> f2f4 -> e5f4 -> g1h3 -> f8c5 -> a1b1 -> d7d5 -> a3b5 -> e8f8
  Reward: -1
  Is Terminal: False

Move: h2h3
  Sequence: h2h3 -> h7h5 -> d2d3 -> b7b6 -> g2g3 -> e7e6 -> g1f3 -> b8a6 -> c1g5 -> c8b7
  Reward: 0
  Is Terminal: False

Move: g2g3
  Sequence: g2g3 -> e7e6 -> c2c3 -> b7b5 -> b1a3 -> g7g6 -> a3b1 -> f8e7 -> d1c2 -> c7c5
  Reward: 0
  Is Terminal: False

Move: f2f3
  Sequence: f2f3 -> b7b6 -> c2c3 -> c8b7 -> f3f4 -> e7e6 -> c3c4 -> b7a6 -> h2h3 -> f8e7
  Reward: 0
  Is Terminal: False

Move: e2e3
  Sequence: e2e3 -> b8a6 -> e1e2 -> c7c5 -> b1c3 -